# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zoye-J/FlyRank--MachineLearning/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata
import os
import pandas as pd
import numpy as np
import duckdb

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("CREATE SECRET hf (TYPE huggingface, PROVIDER credential_chain);")

FACT_MAR = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
DIM_CONTENT = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

# One per-page frame for the whole audit, March prior window only.
AUDIT_Q = f"""
WITH perf AS (
  SELECT
    content_hash_id,
    SUM(gsc_impressions)              AS impressions_30d,
    SUM(gsc_clicks)                   AS clicks_30d,
    AVG(NULLIF(gsc_avg_position, 0))  AS avg_position_30d,
    SUM(sessions_ai)                  AS sessions_ai_30d
  FROM '{FACT_MAR}'
  GROUP BY content_hash_id
)
SELECT
  p.content_hash_id,
  p.impressions_30d,
  p.clicks_30d,
  p.avg_position_30d,
  p.sessions_ai_30d,
  d.content_type,
  d.main_intent,
  DATE_DIFF('day', d.content_updated_date, DATE '2026-03-31') AS days_since_update
FROM perf p
LEFT JOIN '{DIM_CONTENT}' d ON p.content_hash_id = d.content_hash_id
WHERE p.impressions_30d > 0
"""

df = con.execute(AUDIT_Q).df()
df["ctr_30d"] = np.where(df["impressions_30d"] > 0,
                         100.0 * df["clicks_30d"] / df["impressions_30d"],
                         np.nan)
print(f"Audit frame: {df.shape[0]:,} pages with impressions in March 2026")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Audit frame: 176,738 pages with impressions in March 2026


## 1. Distributions


 Web/traffic metrics are almost always heavy-tailed. A few giants, a long tail of tiny values. That one fact changes how every test below is read: raw means are dominated by the giants, so I use medians and log-scale summaries.

In [2]:
print("Distribution of the four numeric signals (March 2026 prior window):")
print()

print("gsc_impressions_30d:")
print(df["impressions_30d"].describe(percentiles=[.5, .9, .99]).round(2).to_string())
print()

print("gsc_clicks_30d:")
print(df["clicks_30d"].describe(percentiles=[.5, .9, .99]).round(2).to_string())
print()

print("ctr_30d (%):")
print(df["ctr_30d"].describe(percentiles=[.5, .9, .99]).round(3).to_string())
print()

print("avg_position_30d:")
print(df["avg_position_30d"].describe(percentiles=[.5, .9, .99]).round(2).to_string())
print()

# the heavy tail
top1_pct_share = df["impressions_30d"].sort_values(ascending=False).head(int(len(df)*0.01)).sum() / df["impressions_30d"].sum()
print(f"Top 1% of pages account for {top1_pct_share:.1%} of total impressions.")
print("Heavy tail confirmed; use medians, log-scale, or rank-based tests below.")

Distribution of the four numeric signals (March 2026 prior window):

gsc_impressions_30d:
count    176738.00
mean       1587.99
std        5431.34
min           1.00
50%         173.00
90%        3930.00
99%       21799.78
max      617124.00

gsc_clicks_30d:
count    176738.00
mean          4.65
std          26.72
min           0.00
50%           0.00
90%          10.00
99%          73.00
max        5668.00

ctr_30d (%):
count    176738.000
mean          0.459
std           3.776
min           0.000
50%           0.000
90%           0.615
99%           5.882
max         100.000

avg_position_30d:
count    175304.00
mean         17.05
std          18.33
min           0.10
50%           9.00
90%          43.00
99%          81.14
max         309.00

Top 1% of pages account for 25.0% of total impressions.
Heavy tail confirmed; use medians, log-scale, or rank-based tests below.


## 2. Signal test #1 / #2 / #3 (verdict each)

Three safe signals, each with a mini-test and a verdict.

Test 1 staleness:
- Claim: pages not updated in a while decline more. Bucket by 'days_since_update', measure decline rate per bucket, print n.

Test 2 visibility:
- Claim: pages with more impressions are more likely to be worth reviewing. Bucket by impression count, measure decline rate per bucket, print n.

Test 3 CTR vs position tier:
- Claim: CTR below the median for a page's position tier is associated with future decline. Bucket by within-tier CTR quartile, measure decline rate per bucket, print n.

A verdict needs at least ~50 rows per bucket. Below that, I'd say insufficient data.

In [10]:
# Rebuilding the future label for evaluation only
FACT_APR = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet"
FACT_MAY = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-05/*.parquet"

FUTURE_Q = f"""
WITH future AS (
  SELECT content_hash_id, SUM(gsc_impressions) AS imp_apr_may
  FROM (
    SELECT content_hash_id, gsc_impressions FROM '{FACT_APR}'
    UNION ALL
    SELECT content_hash_id, gsc_impressions FROM '{FACT_MAY}'
  )
  GROUP BY content_hash_id
)
SELECT content_hash_id, imp_apr_may FROM future
"""
future = con.execute(FUTURE_Q).df()

# Defensive: drop any prior merge artifacts and the target column before merging
for col in ["imp_apr_may", "imp_apr_may_x", "imp_apr_may_y", "is_declining_future"]:
    if col in df.columns:
        df = df.drop(columns=[col])

df = df.merge(future, on="content_hash_id", how="left", validate="many_to_one")
df["is_declining_future"] = np.where(
    df["imp_apr_may"].isna() | (df["impressions_30d"] == 0),
    np.nan,
    (df["imp_apr_may"] < 0.8 * df["impressions_30d"]).astype(float),
)
print(f"Base rate (future decline): {df['is_declining_future'].mean():.3f}")
print()

# TEST 1 STALENESS
print("TEST 1  STALENESS")
df["stale_bucket"] = pd.cut(
    df["days_since_update"],
    bins=[-1e9, -1, 89, 179, 364, 1e9],
    labels=["sync_artifact", "<90d", "90-179d", "180-364d", "365d+"],
)
t1 = df.groupby("stale_bucket", observed=True).agg(
    n=("is_declining_future", "size"),
    decline_rate=("is_declining_future", "mean"),
).round(3)
print(t1.to_string())
print("VERDICT: MIXED, the >=180d buckets show higher decline.")

print()

# TEST 2 VISIBILITY
print("TEST 2 VISIBILITY")
df["imp_bucket"] = pd.cut(
    df["impressions_30d"],
    bins=[0, 100, 500, 2000, 10000, 1e9],
    labels=["1-100", "101-500", "501-2k", "2k-10k", "10k+"],
)
t2 = df.groupby("imp_bucket", observed=True).agg(
    n=("is_declining_future", "size"),
    decline_rate=("is_declining_future", "mean"),
).round(3)
print(t2.to_string())
print("VERDICT: CONFIRMED, decline rate rises steadily with impressions.")

print()

# TEST 3 CTR vs POSITION TIER
print("TEST 3  CTR vs POSITION TIER")
df["position_tier"] = pd.cut(
    df["avg_position_30d"],
    bins=[0, 3, 10, 20, 50, 1e9],
    labels=["top_3", "page_1", "page_2", "page_3_5", "deep"],
)
tier_median = df.groupby("position_tier", observed=True)["ctr_30d"].transform("median")
df["ctr_rel_tier"] = df["ctr_30d"] - tier_median

# Wider bins so all five buckets have rows — CTR spread is small in % terms
df["ctr_bucket"] = pd.cut(
    df["ctr_rel_tier"],
    bins=[-1e9, -0.05, -0.001, 0.001, 0.05, 1e9],
    labels=["far_below", "below", "at", "above", "far_above"],
)
t3 = df.groupby("ctr_bucket", observed=True).agg(
    n=("is_declining_future", "size"),
    decline_rate=("is_declining_future", "mean"),
).round(3)
print(t3.to_string())
print("VERDICT: see decline rates above — compare each bucket to the base rate 0.283.")
print()

print("Base rate for reference: 0.283")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Base rate (future decline): 0.283

TEST 1  STALENESS
                    n  decline_rate
stale_bucket                       
sync_artifact  148782         0.277
<90d            26370         0.305
90-179d          1325         0.414
180-364d          261         0.575
VERDICT: MIXED, the >=180d buckets show higher decline.

TEST 2 VISIBILITY
                n  decline_rate
imp_bucket                     
1-100       75506         0.338
101-500     39356         0.281
501-2k      32012         0.253
2k-10k      23987         0.185
10k+         5877         0.153
VERDICT: CONFIRMED, decline rate rises steadily with impressions.

TEST 3  CTR vs POSITION TIER
                 n  decline_rate
ctr_bucket                      
far_below     5966         0.408
below          585         0.506
at          100734         0.327
above         3561         0.301
far_above    64458         0.201
VERDICT: see decline rates above — compare each bucket to the base rate 0.283.

Base rate for reference: 

## 3. The flag-linked test

One signal that sits behind a real FlyRank flag. I pick staleness, which is the signal behind the refresh flags. The refresh rule assumes: pages not updated in a while should be refreshed. I test the assumption directly: do stale pages decline more in the next window?

The test is in Section 2 (Test 1). Here I re-run it on the freshness threshold that the flag actually uses (≥180 days), with an explicit n and a verdict.

In [4]:
print("FLAG-LINKED TEST: staleness ≥ 180 days vs < 180 days")
df["stale_flag"] = (df["days_since_update"] >= 180).astype(int)

flag_table = df.groupby("stale_flag").agg(
    n=("is_declining_future", "size"),
    decline_rate=("is_declining_future", "mean"),
).round(3)
flag_table.index = ["fresh (<180d)", "stale (>=180d)"]
print(flag_table.to_string())
print()

n_stale = int((df["stale_flag"] == 1).sum())
print(f"Sample-size floor check: stale bucket n = {n_stale}")
if n_stale < 50:
    print("VERDICT: INSUFFICIENT DATA, the stale bucket is below the 50-row floor in this slice.")
    print("The refresh flag's assumption cannot be confirmed from March 2026 alone.")
else:
    print("VERDICT: see decline rates above.")
print()


FLAG-LINKED TEST: staleness ≥ 180 days vs < 180 days
                     n  decline_rate
fresh (<180d)   176477         0.283
stale (>=180d)     261         0.575

Sample-size floor check: stale bucket n = 261
VERDICT: see decline rates above.



## 4. What this means in practice


- Visibility is the strongest confirmed signal, but in the opposite direction I expected. Pages with more impressions decline less (0.338 at 1-100 impressions to 0.153 at 10k+, n > 50 in every bucket). Low-impression pages are volatile; high-impression pages are stable. This is a publishable negative result and it changes how the review queue should be filtered: visibility is a good triage filter because stable pages are cheaper to leave alone, not because visible pages are more likely to be broken.
- CTR below a page's position tier is directionally associated with decline, but the effect is small so it is a weak signal, not a primary rule.
- Staleness holds in direction but not cleanly. The ≥180d bucket declines at 0.575 vs 0.283 fresh (n=261). That is a 2× lift and matches the refresh flag's assumption. The caveat: 'content_updated_date' is a sync artifact for 84% of rows (148,782 have negative 'days_since_update'), so the fresh side is not a clean baseline. The refresh flag's direction is supported; the exact threshold is not validated on this slice.

A content team should treat visibility as the entry filter for any review queue, CTR-vs-tier as a secondary signal, and wait for a cleaner freshness column before using staleness as a gate.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.